# Debug `RxnDataset` on a tiny H2O reaction pair

This notebook builds a tiny reactant/product pair, writes it to `tests/data/`,
loads it with `RxnDataset`, and validates the current dataset contract:
- joint reactant+product graph
- intra-fragment sparse edges only
- `fragment` / `mask` semantics
- node feature layout `h = [pos, one_hot, charge]`


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import torch
from ase import Atoms
from ase.io import write
from torch.utils.data import DataLoader

repo_root = Path.cwd()
if not (repo_root / "dataset").exists() and (repo_root.parent / "dataset").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root.parent))
data_dir = repo_root / "tests" / "data"
data_dir.mkdir(parents=True, exist_ok=True)

from akmcgc.dataset import RxnDataset

print(f"repo_root = {repo_root}")
print(f"data_dir   = {data_dir}")


repo_root = /Users/wx/Desktop/yyxwjq/akmcgc
data_dir   = /Users/wx/Desktop/yyxwjq/akmcgc/tests/data


## Debug Contract Helpers

这些 helper 用来打印 dataset 每个字段的 shape、dtype 和 device。


In [2]:
def tensor_contract(name, value):
    if torch.is_tensor(value):
        return f"{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}"
    return f"{name:16s} value={value!r}"


def print_tensor_contract(title, tensors):
    print(f"\n[{title}]")
    for name, value in tensors.items():
        print(tensor_contract(name, value))


In [3]:
# Build a tiny reaction pair for outer-shell-electron feature checks.
react = Atoms(
    "OHH",
    positions=np.array([[0.00, 0.00, 0.00], [0.96, 0.00, 0.00], [-0.24, 0.93, 0.00]]),
    cell=[16.0, 16.0, 16.0],
    pbc=[False, False, False],
)
react.center(axis=(0, 1, 2), vacuum=7.5)
product = Atoms(
    "HHO",
    positions=np.array([[0.00, 3.00, -0.40], [0.00, 3.00, 0.40], [0.00, -2.00, 0.00]]),
    cell=[16.0, 16.0, 16.0],
    pbc=[False, False, False],
)
product.center(axis=(0, 1, 2), vacuum=7.5)

react_file = data_dir / "h2o_react.extxyz"
product_file = data_dir / "h2o_product.extxyz"
write(react_file, react)
write(product_file, product)


In [4]:
# Optional visualization. Keep disabled for reproducible non-GUI debug runs.

from ase.visualize import view
view(react, viewer="x3d")


In [5]:
view(product, viewer="x3d")

In [16]:
dataset = RxnDataset(
    react_file=str(react_file),
    product_file=str(product_file),
    cutoff=4.0,
    max_neigh=200,
    r_fixed=True,
    r_pbc=True,
    device="cpu",
)

sample = dataset[0]
print(sample['edge_index'])

tensor([[0, 0, 1, 1, 2, 2, 3, 4],
        [2, 1, 0, 2, 0, 1, 4, 3]])


In [10]:
print("Loaded one H2O reaction sample.")
print("Field overview:")
for key, value in sample.items():
    if torch.is_tensor(value):
        print(f"  {key:12s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}")
    else:
        print(f"  {key:12s} {value}")

Loaded one H2O reaction sample.
Field overview:
  h            shape=(6, 122)           dtype=torch.float64  device=cpu
  pos          shape=(6, 3)             dtype=torch.float64  device=cpu
  edge_index   shape=(2, 12)            dtype=torch.int64    device=cpu
  cell_offsets shape=(12, 3)            dtype=torch.float64  device=cpu
  neighbors    shape=(1,)               dtype=torch.int64    device=cpu
  fragment     shape=(6,)               dtype=torch.int64    device=cpu
  mask         shape=(6,)               dtype=torch.int64    device=cpu
  cell         shape=(2, 3, 3)          dtype=torch.float64  device=cpu
  pbc          shape=(2, 3)             dtype=torch.bool     device=cpu
  n_is         3
  n_fs         3


## Sample Tensor Contract

单个样本从 `RxnDataset.__getitem__` 出来后，每个字段的维度和 dtype 如下。


In [7]:
print_tensor_contract('RxnDataset.__getitem__ sample', sample)



[RxnDataset.__getitem__ sample]
h                shape=(6, 122)           dtype=torch.float64  device=cpu
pos              shape=(6, 3)             dtype=torch.float64  device=cpu
edge_index       shape=(2, 12)            dtype=torch.int64    device=cpu
cell_offsets     shape=(12, 3)            dtype=torch.float64  device=cpu
neighbors        shape=(1,)               dtype=torch.int64    device=cpu
fragment         shape=(6,)               dtype=torch.int64    device=cpu
mask             shape=(6,)               dtype=torch.int64    device=cpu
cell             shape=(2, 3, 3)          dtype=torch.float64  device=cpu
pbc              shape=(2, 3)             dtype=torch.bool     device=cpu
n_is             value=3
n_fs             value=3


In [8]:
# Validate collate_fn behavior on two copies of the same sample.
batch = RxnDataset.collate_fn([sample, sample])
n_total = sample["n_is"] + sample["n_fs"]

assert batch["h"].shape == (2 * n_total, 122)
assert batch["pos"].shape == (2 * n_total, 3)
assert batch["fragment"].shape == (2 * n_total,)
assert batch["mask"].shape == (2 * n_total,)
assert torch.equal(batch["mask"], torch.tensor([0] * n_total + [1] * n_total))

src_b, dst_b = batch["edge_index"]
assert torch.all(batch["mask"][src_b] == batch["mask"][dst_b])
assert torch.all(batch["fragment"][src_b] == batch["fragment"][dst_b])

print("Collated two H2O samples into one disconnected joint graph batch.")
print("Batch field overview:")
for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"  {key:12s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}")
    else:
        print(f"  {key:12s} {value}")
print("Batch checks passed.")


Collated two H2O samples into one disconnected joint graph batch.
Batch field overview:
  h            shape=(12, 122)          dtype=torch.float64  device=cpu
  pos          shape=(12, 3)            dtype=torch.float64  device=cpu
  edge_index   shape=(2, 24)            dtype=torch.int64    device=cpu
  cell_offsets shape=(24, 3)            dtype=torch.float64  device=cpu
  neighbors    shape=(2,)               dtype=torch.int64    device=cpu
  fragment     shape=(12,)              dtype=torch.int64    device=cpu
  mask         shape=(12,)              dtype=torch.int64    device=cpu
  cell         shape=(2, 2, 3, 3)       dtype=torch.float64  device=cpu
  pbc          shape=(2, 2, 3)          dtype=torch.bool     device=cpu
  n_is         shape=(2,)               dtype=torch.int64    device=cpu
  n_fs         shape=(2,)               dtype=torch.int64    device=cpu
Batch checks passed.


## Batched Tensor Contract

`RxnDataset.collate_fn` 会把多个样本拼成一个 disconnected joint graph batch。这里检查 batch 后每个字段的维度和 dtype。


In [9]:
print_tensor_contract('RxnDataset.collate_fn batch', batch)



[RxnDataset.collate_fn batch]
h                shape=(12, 122)          dtype=torch.float64  device=cpu
pos              shape=(12, 3)            dtype=torch.float64  device=cpu
edge_index       shape=(2, 24)            dtype=torch.int64    device=cpu
cell_offsets     shape=(24, 3)            dtype=torch.float64  device=cpu
neighbors        shape=(2,)               dtype=torch.int64    device=cpu
fragment         shape=(12,)              dtype=torch.int64    device=cpu
mask             shape=(12,)              dtype=torch.int64    device=cpu
cell             shape=(2, 2, 3, 3)       dtype=torch.float64  device=cpu
pbc              shape=(2, 2, 3)          dtype=torch.bool     device=cpu
n_is             shape=(2,)               dtype=torch.int64    device=cpu
n_fs             shape=(2,)               dtype=torch.int64    device=cpu
